In [ ]:

import os
import numpy as np
import datetime
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, callbacks

#### GPU memory optimization


In [ ]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Enabled memory growth for GPUs")
    except RuntimeError as e:
        print(f"Could not enable memory growth: {e}")

## User defined parameters

In [ ]:

IMG_SIZE = 256

TRAIN_INPUT_DIR = 'Your path to the train input images (folder)'
TRAIN_LABEL_DIR = 'Your path to the train label images (folder)'
TEST_INPUT_DIR = 'Your path to the test input images (folder)'
TEST_LABEL_DIR = 'Your path to the test label images (folder)'

# log directory for tensorboard
LOG_DIR = f"your log dir/{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"
MODEL_SAVE_PATH = 'your model save path'

#### Pretreat

In [ ]:

def load_images(image_dir, target_size=(IMG_SIZE, IMG_SIZE)):
    images = []
    for filename in os.listdir(image_dir):
        img_path = os.path.join(image_dir, filename)
        try:
            img = tf.keras.preprocessing.image.load_img(img_path, color_mode='grayscale', target_size=target_size)
            img = tf.keras.preprocessing.image.img_to_array(img) / 255.0  
            images.append(img)
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
    return np.array(images)

#### Load 

In [ ]:

print("Loading training and testing data...")
train_images = load_images(TRAIN_INPUT_DIR)
train_labels = load_images(TRAIN_LABEL_DIR)
test_images = load_images(TEST_INPUT_DIR)
test_labels = load_images(TEST_LABEL_DIR)



#### Split Tr Val Te

In [ ]:

train_images, val_images, train_labels, val_labels = train_test_split(
    train_images, train_labels, test_size=0.2, random_state=42
)
print("Data loaded successfully!")

#### Net

In [ ]:


def build_model(input_shape=(256, 256, 1)):
    inputs = layers.Input(shape=input_shape)

    # Encoder part (Downsampling)
    # First Block (256x256)
    x1 = layers.Conv2D(64, (3, 3), padding='same')(inputs)
    x1 = layers.BatchNormalization()(x1)
    x1 = layers.Activation('elu')(x1)
    x2 = layers.Conv2D(64, (3, 3), padding='same')(x1)
    x2 = layers.BatchNormalization()(x2)
    x2 = layers.Activation('elu')(x2)
    skip1 = x2  # Skip connection (to be used in decoder)
    x2 = layers.MaxPooling2D(pool_size=(2, 2))(x2)
    x2 = layers.Dropout(0.3)(x2)

    # Second Block (128x128)
    x3 = layers.Conv2D(128, (3, 3), padding='same')(x2)
    x3 = layers.BatchNormalization()(x3)
    x3 = layers.Activation('elu')(x3)
    x4 = layers.Conv2D(128, (3, 3), padding='same')(x3)
    x4 = layers.BatchNormalization()(x4)
    x4 = layers.Activation('elu')(x4)
    skip2 = x4  # Skip connection (to be used in decoder)
    x4 = layers.MaxPooling2D(pool_size=(2, 2))(x4)
    x4 = layers.Dropout(0.3)(x4)

    # Third Block (64x64)
    x5 = layers.Conv2D(256, (3, 3), padding='same')(x4)
    x5 = layers.BatchNormalization()(x5)
    x5 = layers.Activation('elu')(x5)
    x6 = layers.Conv2D(256, (3, 3), padding='same')(x5)
    x6 = layers.BatchNormalization()(x6)
    x6 = layers.Activation('elu')(x6)
    skip3 = x6  # Skip connection (to be used in decoder)
    x6 = layers.MaxPooling2D(pool_size=(2, 2))(x6)
    x6 = layers.Dropout(0.3)(x6)

    # Fourth Block (32x32)
    x7 = layers.Conv2D(512, (3, 3), padding='same')(x6)
    x7 = layers.BatchNormalization()(x7)
    x7 = layers.Activation('elu')(x7)
    x8 = layers.Conv2D(512, (3, 3), padding='same')(x7)
    x8 = layers.BatchNormalization()(x8)
    x8 = layers.Activation('elu')(x8)
    skip4 = x8  # Skip connection (to be used in decoder)
    x8 = layers.MaxPooling2D(pool_size=(2, 2))(x8)
    x8 = layers.Dropout(0.3)(x8)

    # Bottom Block (16x16)
    x9 = layers.Conv2D(1024, (3, 3), padding='same')(x8)
    x9 = layers.BatchNormalization()(x9)
    x9 = layers.Activation('elu')(x9)
    x10 = layers.Conv2D(1024, (3, 3), padding='same')(x9)
    x10 = layers.BatchNormalization()(x10)
    x10 = layers.Activation('elu')(x10)

    # Decoder part (Upsampling)
    # Upsample to 32x32
    x11 = layers.Conv2DTranspose(1024, (3, 3), strides=(2, 2), padding='same')(x10)
    x11 = layers.Concatenate()([x11, skip4])  # Skip connection from Encoder
    x11 = layers.Conv2D(512, (3, 3), padding='same')(x11)
    x11 = layers.BatchNormalization()(x11)
    x11 = layers.ReLU()(x11)
    x11 = layers.Dropout(0.3)(x11)

    # Upsample to 64x64
    x12 = layers.Conv2DTranspose(512, (3, 3), strides=(2, 2), padding='same')(x11)
    x12 = layers.Concatenate()([x12, skip3])  # Skip connection from Encoder
    x12 = layers.Conv2D(256, (3, 3), padding='same')(x12)
    x12 = layers.BatchNormalization()(x12)
    x12 = layers.ReLU()(x12)
    x12 = layers.Dropout(0.3)(x12)

    # Upsample to 128x128
    x13 = layers.Conv2DTranspose(256, (3, 3), strides=(2, 2), padding='same')(x12)
    x13 = layers.Concatenate()([x13, skip2])  # Skip connection from Encoder
    x13 = layers.Conv2D(128, (3, 3), padding='same')(x13)
    x13 = layers.BatchNormalization()(x13)
    x13 = layers.ReLU()(x13)
    x13 = layers.Dropout(0.3)(x13)

    # Upsample to 256x256
    x14 = layers.Conv2DTranspose(128, (3, 3), strides=(2, 2), padding='same')(x13)
    x14 = layers.Concatenate()([x14, skip1])  # Skip connection from Encoder
    x14 = layers.Conv2D(64, (3, 3), padding='same')(x14)
    x14 = layers.BatchNormalization()(x14)
    x14 = layers.ReLU()(x14)
    x15 = layers.Conv2D(64, (3, 3), padding='same')(x15)
    x15 = layers.BatchNormalization()(x15)
    x15 = layers.ReLU()(x15)
    x15 = layers.Dropout(0.3)(x15)

    # Final Conv Layer
    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(x14)  # Sigmoid for binary classification

    # Create model
    model = models.Model(inputs, outputs)

    return model

# Build the model
model = build_model()
model.summary()


#### Loss

In [ ]:
def dice_loss(y_true, y_pred):
    """
    Dice Loss for pixel-wise segmentation.
    Computes the Dice Coefficient loss.
    """
    smooth = 1.0  # To avoid division by zero
    y_true_f = tf.reshape(y_true, [-1])  # Flatten ground truth
    y_pred_f = tf.reshape(y_pred, [-1])  # Flatten characterizeions
    intersection = tf.reduce_sum(y_true_f * y_pred_f)  # Intersection
    return 1 - (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def combined_loss(y_true, y_pred):
    """
    Combined loss using Binary Cross-Entropy, Dice Loss, and SSIM.
    """
    # Binary Cross-Entropy
    bce = tf.keras.losses.BinaryCrossentropy()(y_true, y_pred)
    
    # Dice Loss
    dice = dice_loss(y_true, y_pred)
    
    # SSIM Loss (1 - SSIM)
    ssim_loss = 1 - tf.image.ssim(y_true, y_pred, max_val=1.0)
    
    return 0.25 * bce + 0.25 * dice + 0.5 * ssim_loss  # Adjust weights as needed

def psnr_metric(y_true, y_pred):
    return tf.image.psnr(y_true, y_pred, max_val=1.0)

def ssim_metric(y_true, y_pred):
    return tf.image.ssim(y_true, y_pred, max_val=1.0)

In [ ]:
# model.compile
model = build_model(IMG_SIZE)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=combined_loss,
    metrics=[psnr_metric, ssim_metric]
)

In [ ]:
# callbacks
tensorboard_callback = callbacks.TensorBoard(log_dir=LOG_DIR, histogram_freq=1)
lr_scheduler = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-8)
model_checkpoint = callbacks.ModelCheckpoint(
    MODEL_SAVE_PATH, monitor='val_loss', save_best_only=True, mode='min', verbose=1
)

In [80]:
# train
history = model.fit(
    train_images, train_labels,
    validation_data=(val_images, val_labels),
    epochs=200,
    batch_size=4,
    callbacks=[tensorboard_callback, lr_scheduler, model_checkpoint]
)

In [ ]:
# save model
final_model_path = 'F:/B208/DeepLearning/Model/5_segmentation_model.h5'
model.save(final_model_path)
print(f"Model saved to {final_model_path}")

In [ ]:
# Tensorboard
print(f"To visualize U-Net logs, run:\n    tensorboard --logdir={LOG_DIR}")